# INC data check

Work through four questions: pathology without neck dissection, missing
observations, consistency, and the meaning of the recorded consensus.

The calculations are shared in `scripts/data_checks.py`; this notebook selects
the INC workbook and presents its results. **Run all cells** to refresh the report.
All patients remain included. Expand a code cell, evidence table or settings panel
when you need more detail.

An observation is one **patient–side–level** value. The main-level summaries use
**I, II, III, IV, V and VII on both sides**. Unknown values remain unknown.

In [1]:
DATASET_NAME = "INC"
WORKBOOK_PATH = "updated_datasets/INC_update.xlsx"

In [2]:
from pathlib import Path
import sys

import pandas as pd
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "updated_datasets").exists():
    PROJECT_DIR = PROJECT_DIR / "Mexico_data_analysis"

if str(PROJECT_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR.resolve()))

import importlib
from scripts import data_checks as checks
importlib.reload(checks)
from scripts.notebook_display import SOURCE_LABELS, comparison_view, show_note, show_table

DATA_FILE = PROJECT_DIR / WORKBOOK_PATH
source_data = pd.read_excel(DATA_FILE, header=[0, 1, 2])
analysis = checks.analyse_dataset(source_data)
raw = analysis["raw"]
without = "max_llh_without_pathology"
with_pathology = "max_llh_with_pathology"
show_note(f"{DATASET_NAME} · {len(raw)} patients", f"Source: {DATA_FILE.name}")

## 1. Pathology without neck dissection

Find patients with at least one recorded pathology result whose neck-dissection
flag is False or missing. Both positive and negative pathology count, including
results recorded only at sublevels.

Read each patient's recorded levels alongside the centre's comments. A biopsy
can explain pathology without a neck dissection; the finding is a request for
context, not an automatic correction of the flag.

In [3]:
pathology_without_neck_dissection = analysis["pathology_without_neck_dissection"]
show_note(
    f"{len(pathology_without_neck_dissection)} of {len(raw)} patients have pathology without dissection marked True",
    "This can reflect a biopsy. Review the centre's comments before changing a flag.",
)

def compact_pathology(text):
    by_side = {}
    for item in text.split("; "):
        side, observation = item.split(" ", 1)
        level, value = observation.split("=", 1)
        by_side.setdefault(side, []).append(level + (" +" if value == "positive" else " −"))
    return "; ".join(f"{side}: {', '.join(values)}" for side, values in by_side.items())

pathology_view = pathology_without_neck_dissection.copy()
pathology_view["neck_dissection"] = pathology_view["neck_dissection"].map(
    {True: "Yes", False: "No"}
).fillna("Unknown")
pathology_view["pathology_observations"] = pathology_view["pathology_observations"].map(compact_pathology)
show_table(pathology_view.rename(columns={
    "patient_id": "Patient", "neck_dissection": "Neck dissection",
    "pathology_observations": "Pathology (+ / −)", "comments": "Centre's comments",
}))

Patient,Neck dissection,Pathology (+ / −),Centre's comments
INC-6,No,"ipsi: I −, Ia −, Ib −, II +, IIa +, IIb +, III −, IV −, V −, VII −; contra: I −, Ia −, Ib −, II −, IIa −, IIb −, III −, IV −, V −, VII −",—


## 2. Missing observations

For each source, count patients with any data and known versus missing main-level
values. There are **12 possible values per patient**: six levels on each side.
An absent modality block is marked separately from a block with missing entries.

Coverage describes the information available for later comparisons. It does not
imply that every patient should have undergone every test. Missing observations
are never counted as negative results.

In [4]:
missing_observations = analysis["missing_observations"]
missing_view = missing_observations.copy()
missing_view["Patients with data"] = missing_view["patients_with_data"].map(lambda n: f"{n}/{len(raw)}")
missing_view["Missing values"] = missing_view["missing_pairs"].map(
    lambda n: f"{n}/{len(raw) * 12}"
)
missing_view["modality"] = missing_view["modality"].replace({"diagnostic_consensus": "Recorded consensus"})
missing_view.loc[missing_view["columns_present"].eq(0), "modality"] += " (absent)"
show_table(missing_view[["modality", "Patients with data", "Missing values", "coverage_pct"]].rename(
    columns={"modality": "Source", "coverage_pct": "Coverage"}
), percentages=["Coverage"])

Source,Patients with data,Missing values,Coverage
Recorded consensus,73/73,0/876,100.0%
pathology,1/73,864/876,1.4%
CT,38/73,420/876,52.1%
MRI,10/73,756/876,13.7%
PET,19/73,648/876,26.0%
FNA,0/73,876/876,0.0%
pCT (absent),0/73,876/876,0.0%


## 3. Consistency

Check I against Ia/Ib and II against IIa/IIb within each modality. A positive
child requires a positive parent; two negative children require a negative
parent. One negative child with an unknown sibling is insufficient.

Also run the same reduced checks for identity, valid values, N-stage and node
counts used in the original notebook. Findings are grouped below. Invalid binary
entries are reported here and treated as unknown in comparisons and reconstruction.
Consensus–pathology disagreements are examined separately in section 4.

In [5]:
hierarchy_issues = analysis["hierarchy_issues"]
sanity_issues = analysis["sanity_issues"]
consistency_summary = analysis["consistency_summary"]

total_flags = len(hierarchy_issues) + len(sanity_issues)
show_note(
    "No consistency flags" if total_flags == 0 else f"{total_flags} consistency findings",
    f"Main/sublevel conflicts: {len(hierarchy_issues)}. Reduced sanity findings: {len(sanity_issues)}.",
)
if not hierarchy_issues.empty:
    show_table(hierarchy_issues, details="Main/sublevel findings")
if not sanity_issues.empty:
    show_table(sanity_issues[["patient_id", "check", "category", "details"]],
               details="Reduced sanity-check findings")

## 4. What does the recorded consensus represent?

### Recorded consensus versus each source

Compare the recorded consensus with pathology, CT, MRI, PET, FNA and pCT at the
same patient, side and main level. Only pairs with **both values known** enter
the denominator; the patient column counts people with a disagreement.

Agreement percentages include both positive and negative agreement. High overall
agreement can coexist with important positive-case disagreements, so the case
evidence below remains essential. No comparable pairs means no percentage.

In [6]:
recorded_vs_modalities = analysis["recorded_vs_modalities"]
pathology_result = recorded_vs_modalities.iloc[0]
show_note(
    f"Consensus–pathology disagreements: {int(pathology_result['disagreements'])} / "
    f"{int(pathology_result['compared_pairs'])} known pairs",
    f"Patients affected: {int(pathology_result['patients_affected'])}. Individual cases appear below.",
)
show_table(comparison_view(recorded_vs_modalities, ["Pathology", "CT", "MRI", "PET", "FNA", "pCT"]),
           percentages=["Agreement"])

Comparison,Known pairs,Disagreements,Patients affected,Agreement
Pathology,12,0,0,100.0%
CT,456,13,10,97.1%
MRI,120,7,4,94.2%
PET,228,6,5,97.4%
FNA,0,0,0,—
pCT,0,0,0,—


### Reconstruct consensus with and without pathology

**Without pathology:** CT, MRI, PET, pCT and FNA, where available. **With pathology:**
the same inputs plus pathology. Both use `max_llh`; the recorded consensus is
excluded from both, and FNA remains included.


<details>
<summary>Reconstruction method notes</summary>

Both runs use the same installed lydata defaults and I/II subdivisions. The source observations are preserved. Parent or child labels can be inferred; otherwise, no input means an unknown label.

Default pathology sensitivity and specificity are 1.0. Agreement of a reconstruction containing pathology with that same input is expected by construction.

</details>

In [7]:
modality_settings = analysis["modality_settings"]
reconstruction_summary = analysis["reconstruction_summary"]
show_table(comparison_view(reconstruction_summary, [
    "Recorded vs without pathology", "Recorded vs with pathology", "Without vs with pathology",
]), percentages=["Agreement"])

settings_view = modality_settings.rename(columns={
    "modality": "Modality", "available": "Present", "sensitivity": "Sensitivity",
    "specificity": "Specificity",
})
settings_view["Present"] = settings_view["Present"].map({True: "Yes", False: "No"})
show_table(settings_view, details="Model assumptions: sensitivity and specificity")

Comparison,Known pairs,Disagreements,Patients affected,Agreement
Recorded vs without pathology,804,26,19,96.8%
Recorded vs with pathology,816,26,19,96.8%
Without vs with pathology,804,0,0,100.0%


Modality,Present,Sensitivity,Specificity
CT,Yes,0.81,0.76
MRI,Yes,0.81,0.63
PET,Yes,0.79,0.86
pCT,No,0.81,0.86
FNA,Yes,0.8,0.98
pathology,Yes,1,1


### Which reconstruction agrees with the recorded consensus?

First identify known labels that change when pathology is added. Among those
with known pathology and recorded consensus, count how often the recorded
consensus agrees with each reconstruction. This focuses on the observations
that can distinguish a clinical consensus from one incorporating pathology.

The main case table includes disagreements involving pathology or the
reconstructed consensuses. Ordinary differences with an individual modality
appear only in the expandable full evidence table.
The expanded pathology comparison uses the same known pairs for all three
consensuses. Agreement alone cannot confirm the intended definition or timing.

**+** positive · **−** negative · **—** unknown.

For I and II, **Hierarchy** explains any reconstructed parent change and
shows the resulting sublevels. Expand the parent/sublevel evidence to
compare raw I/Ia/Ib or II/IIa/IIb with each reconstruction before and
after hierarchy reconciliation. The modality values in the main table
are raw observations.


In [8]:
matched_pathology_summary = analysis["matched_pathology_summary"]
reconstruction_changed = analysis["reconstruction_changed"]
interpretation_summary = analysis["interpretation_summary"]
show_note(
    f"Labels changed by pathology: {int(reconstruction_changed.to_numpy().sum())}",
    f"Patients affected: {int(reconstruction_changed.any(axis=1).sum())}.",
)
interpretation_view = pd.DataFrame({
    "Reconstruction": ["Without pathology", "With pathology"],
    "Agrees with recorded consensus on changed pairs": [
        f"{row.compared_pairs - row.disagreements}/{row.compared_pairs}"
        if row.compared_pairs else "—"
        for row in interpretation_summary.itertuples(index=False)
    ],
})
show_table(interpretation_view)

# Keep every original evidence value, using short labels only in the display.
disagreements = analysis["disagreements"]
all_disagreements = analysis["all_disagreements"]
evidence_view = disagreements[["patient_id", "recorded_consensus", "pathology", without, with_pathology]].copy()
evidence_view.insert(1, "Location", disagreements["side"] + " " + disagreements["level"])
evidence_view["Raw modalities"] = [
    ", ".join(f"{name} {'+' if row[name] else '−'}" for name in ["CT", "MRI", "PET", "FNA"]
              if pd.notna(row[name]))
    for _, row in disagreements.iterrows()
]
evidence_view["Hierarchy"] = disagreements["hierarchy_changes"]
evidence_view = evidence_view.rename(columns={
    "patient_id": "Patient", "recorded_consensus": "Recorded", "pathology": "Pathology",
    without: "Without path.", with_pathology: "With path.",
})
show_table(evidence_view, binary=["Recorded", "Pathology", "Without path.", "With path."])

evidence_columns = ["patient_id", "side", "level", "recorded_consensus", "pathology",
                    "CT", "MRI", "PET", "FNA", without, with_pathology, "comments"]
full_evidence = all_disagreements[evidence_columns].rename(columns={
    "patient_id": "Patient", "side": "Side", "level": "Level", "comments": "Centre's comments",
    **SOURCE_LABELS,
})
show_table(full_evidence, details="All disagreements, including individual modalities, and centre comments",
           binary=["Recorded consensus", "Pathology", "CT", "MRI", "PET", "FNA",
                   "Without pathology", "With pathology"])
show_table(comparison_view(matched_pathology_summary, [
    "Recorded consensus vs pathology", "Without pathology vs pathology", "With pathology vs pathology",
]), percentages=["Agreement"], details="Agreement with pathology on the same known pairs")

sublevel_evidence = analysis["hierarchy_evidence"].copy()
if not sublevel_evidence.empty:
    sublevel_evidence["source"] = sublevel_evidence["source"].replace(SOURCE_LABELS)
    show_table(sublevel_evidence.rename(columns={
        "patient_id": "Patient", "location": "Location", "source": "Source",
        "stage": "Stage", "parent_value": "Parent", "sublevels": "Sublevels",
    }), details="Parent and sublevel evidence: before and after hierarchy", binary=["Parent"])


Reconstruction,Agrees with recorded consensus on changed pairs
Without pathology,—
With pathology,—


Patient,Location,Recorded,Pathology,Without path.,With path.,Raw modalities,Hierarchy
INC-5,ipsi IV,+,<NA>,−,−,CT −,—
INC-15,ipsi I,+,<NA>,−,−,MRI −,—
INC-15,ipsi III,−,<NA>,+,+,MRI +,—
INC-17,contra I,−,<NA>,+,+,MRI +,—
INC-23,contra II,+,<NA>,−,−,CT −,—
INC-23,contra III,+,<NA>,−,−,CT −,—
INC-24,contra III,−,<NA>,+,+,PET +,—
INC-24,contra IV,−,<NA>,+,+,PET +,—
INC-30,ipsi III,−,<NA>,+,+,CT +,—
INC-31,contra IV,+,<NA>,−,−,MRI −,—


Patient,Side,Level,Recorded consensus,Pathology,CT,MRI,PET,FNA,Without pathology,With pathology,Centre's comments
INC-5,ipsi,IV,+,<NA>,−,<NA>,<NA>,<NA>,−,−,—
INC-15,ipsi,I,+,<NA>,<NA>,−,<NA>,<NA>,−,−,—
INC-15,ipsi,III,−,<NA>,<NA>,+,<NA>,<NA>,+,+,—
INC-17,contra,I,−,<NA>,<NA>,+,<NA>,<NA>,+,+,—
INC-23,contra,II,+,<NA>,−,<NA>,<NA>,<NA>,−,−,—
INC-23,contra,III,+,<NA>,−,<NA>,<NA>,<NA>,−,−,—
INC-24,contra,III,−,<NA>,<NA>,<NA>,+,<NA>,+,+,—
INC-24,contra,IV,−,<NA>,<NA>,<NA>,+,<NA>,+,+,—
INC-30,ipsi,III,−,<NA>,+,<NA>,<NA>,<NA>,+,+,—
INC-31,contra,IV,+,<NA>,<NA>,−,<NA>,<NA>,−,−,—


Comparison,Known pairs,Disagreements,Patients affected,Agreement
Recorded consensus vs pathology,0,0,0,—
Without pathology vs pathology,0,0,0,—
With pathology vs pathology,0,0,0,—


Patient,Location,Source,Stage,Parent,Sublevels
INC-15,ipsi I,Recorded consensus,Recorded,+,"Ia −, Ib +"
INC-15,ipsi I,MRI,Recorded,−,"Ia −, Ib −"
INC-15,ipsi I,Without pathology,Before hierarchy,−,"Ia −, Ib −"
INC-15,ipsi I,Without pathology,After hierarchy,−,"Ia −, Ib −"
INC-15,ipsi I,With pathology,Before hierarchy,−,"Ia −, Ib −"
INC-15,ipsi I,With pathology,After hierarchy,−,"Ia −, Ib −"
INC-17,contra I,Recorded consensus,Recorded,−,"Ia −, Ib −"
INC-17,contra I,MRI,Recorded,+,"Ia −, Ib +"
INC-17,contra I,Without pathology,Before hierarchy,+,"Ia −, Ib +"
INC-17,contra I,Without pathology,After hierarchy,+,"Ia −, Ib +"


### N-stage versus involvement

Compare the recorded N-stage with involvement from the recorded consensus,
both reconstructions and pathology. Flag **N0 with a positive level** and
**N+ without an observed positive**. Distinguish complete negative coverage
from partial negative or absent observations.

The expanded overview lists positive levels on each side for every patient.
The N-stage basis remains unconfirmed unless its own field or prefix specifies
it; a T-stage prefix does not establish an N-stage basis. These are N0/N+
review prompts, not a reconstruction of exact TNM substages. Different sources
may also describe different assessments.

In [9]:
n_stage_details = analysis["n_stage_details"]
n_stage_overview = analysis["n_stage_overview"]
n_stage_review = analysis["n_stage_review"]
review_patients = n_stage_review["patient_id"].unique()
show_note(
    f"N-stage review: {len(review_patients)} " + ("patient" if len(review_patients) == 1 else "patients"),
    ", ".join(map(str, review_patients)) if len(review_patients) else "No N0/N+ review findings.",
)
if not n_stage_review.empty:
    stage_review_view = n_stage_review[[
        "patient_id", "n_stage", "source", "known_pairs", "assessment"
    ]].copy()
    stage_review_view["source"] = stage_review_view["source"].replace(SOURCE_LABELS)
    stage_review_view["known_pairs"] = stage_review_view["known_pairs"].map(lambda n: f"{n}/12")
    stage_review_view["assessment"] = stage_review_view["assessment"].str.removeprefix("Review: ")
    show_table(stage_review_view.rename(columns={
        "patient_id": "Patient", "n_stage": "N-stage", "source": "Source",
        "known_pairs": "Observed", "assessment": "Review finding",
    }))
show_table(n_stage_overview.rename(columns={
    "patient_id": "Patient", "n_stage": "N-stage", "stage_basis": "N-stage basis", **SOURCE_LABELS,
}), details="All patients: N-stage and involved levels on each side")

Patient,N-stage,N-stage basis,Recorded consensus,Without pathology,With pathology,Pathology
INC-1,1,unconfirmed,ipsi: II; contra: none observed,ipsi: II; contra: none observed,ipsi: II; contra: none observed,No observations
INC-2,3,unconfirmed,ipsi: II; contra: II,ipsi: II; contra: II,ipsi: II; contra: II,No observations
INC-3,2,unconfirmed,ipsi: II; contra: II,ipsi: II; contra: II,ipsi: II; contra: II,No observations
INC-4,2b,unconfirmed,"ipsi: II, III; contra: none observed","ipsi: II, III; contra: none observed","ipsi: II, III; contra: none observed",No observations
INC-5,2c,unconfirmed,"ipsi: II, III, IV; contra: II","ipsi: II, III; contra: II","ipsi: II, III; contra: II",No observations
INC-6,3,unconfirmed,ipsi: II; contra: none observed,No observations,ipsi: II; contra: none observed,ipsi: II; contra: none observed
INC-7,0,unconfirmed,No positives (12/12 observed),No positives (12/12 observed),No positives (12/12 observed),No observations
INC-8,1,unconfirmed,ipsi: II; contra: none observed,ipsi: II; contra: none observed,ipsi: II; contra: none observed,No observations
INC-9,2,unconfirmed,ipsi: V; contra: none observed,ipsi: V; contra: none observed,ipsi: V; contra: none observed,No observations
INC-10,1,unconfirmed,"ipsi: II, III; contra: none observed","ipsi: II, III; contra: none observed","ipsi: II, III; contra: none observed",No observations
